# Problema Direto Estacionário (*Vanilla*-PINN)


Este notebbok tem como objetivo explorar a solução de um problema direto estacionário com uma PINN do tipo vanilla (a formulação original).

**Autor:** Edélio Gabriel Magalhães de Jesus.

## Definição do problema

Um problema direto, como discutido no *notebook* introdutório (ver `00_introduction.ipynb`), busca encontrar os efeitos a partir das causas — conhecidos os parâmetros que governam o fenômeno e as condições impostas nas fronteiras, determina-se a solução no interior do domínio.

Para exemplificar esse procedimento, trabalharemos com a **equação de Laplace 2D** no domínio unitário $\Omega = [0,1] \times [0,1]$, com as seguintes condições de contorno:

$$
\begin{cases}
u(x, 0) = 0 \\ 
u(x, 1) = \sin(\pi x) \\
u(0, y) = 0 \\
u(1, y) = 0
\end{cases}
$$

Geometricamente, três bordas do domínio estão fixas em zero, enquanto a borda superior possui um perfil prescrito — nulo nas extremidades e com um pico no centro. A solução analítica para esse sistema é conhecida e será utilizada como referência para validação:

$$
u(x, y) = \frac{\sinh(\pi y)}{\sinh(\pi)}\sin(\pi x)
$$

---
### `Requisitos teóricos`

#### O que é a equação de Laplace?

A equação de Laplace é uma equação diferencial parcial elíptica de ampla relevância em ciência e engenharia — ela descreve fenômenos tão distintos quanto o potencial gravitacional, o potencial elétrico e o escoamento irrotacional de fluidos. A teoria geral de suas soluções é conhecida como **teoria do potencial** [[ref]](#laplace-eq).

Ela é definida como:

$$
\Delta u = 0
$$

onde $\Delta$ é o **operador laplaciano**, dado no caso bidimensional por:

$$
\Delta u = \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} = 0
$$

Uma possível  interpretação geométrica: $\Delta u = 0$ impõe que a curvatura média da superfície $u(x,y)$ é nula em todo o interior do domínio. Isso significa que a solução não pode ter máximos ou mínimos locais no interior — ela é completamente determinada pelos valores prescritos na fronteira. Esse resultado é conhecido como o **Princípio do Máximo** [[ref]](#laplace-eq).

Vale notar ainda que, quando o lado direito é não nulo — isto é, quando há uma fonte no interior do domínio — a equação passa a ser chamada de **equação de Poisson**:

$$
\Delta u = f(x, y)
$$

que exploraremos em outros exemplos.

---

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex01_pinn_direct_stacionary_vannila.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

### **Importações necessárias**

In [13]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_stationary
from ex01_pinn_direct_stacionary_vannila import analytical_solution, evaluate, train
from plot_utils import plot_loss, plot_heatmaps, plot_profiles, plot_points_stationary

### **Parâmetros fixos do problema**

Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados nos usados por Baty em "***A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics.***"[[ref]](#hands-on-paper).

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS = 2
N_OUTPUTS = 1
N_HIDDEN = 16
N_LAYERS = 4
ACTIVATION = nn.Tanh

# Parâmetros do problema
LB = [0,0]
UB = [1,1]

BC_FNS = {
    'bottom': lambda x: torch.zeros_like(x),
    'top':    lambda x: torch.sin(torch.pi * x),
    'left':   lambda x: torch.zeros_like(x),
    'right':  lambda x: torch.zeros_like(x)
}

# Parâmetros de amostragem
N_COLLOC = 500
N_BC = 30

# Parâmetros do treinamento
W_DATA = 1.0
W_PDE = 1.0
N_EPOCHS = 50000
LR = 1e-4


Usando: cpu


Um ponto importante de destacar aqui é a definição da função de ativação. Como o problema envolve valores positivos e negativos, não é adequado usar funções como a Sigmoid, que é limitada ao intervalo (0,1), pois isso restringe artificialmente a saída da rede.

Além disso, nosso problema exige diferenciabilidade de segunda ordem — devido ao operador laplaciano — o que implica que a função de ativação deve ser pelo menos $C^2$. Nesse sentido, não podemos usar a ReLU, pois ela não é diferenciável em 0 e tem segunda derivada nula quase em todo o domínio, o que dificulta a representação de soluções suaves.

Portanto, escolhemos a Tanh, que é suave ($C^\infty$), assume valores positivos e negativos e consegue representar melhor as variações suaves exigidas pela equação de Laplace.

Note ainda a quantidade irrisória de pontos amostrados: 500 internos e 30 por borda, totalizando 620 pontos no domínio contínuo.

### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION)

### **Amostragem dos pontos**

In [5]:
X_BC, U_BC = sample_boundary_rectangular_stationary(N_BC, LB, UB, BC_FNS, DEVICE)
X_COLLOC = sample_collocation_rectangular(N_COLLOC, LB, UB, DEVICE)

### **Instanciando o otimizador**

In [6]:
OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LR)

### **Treinamento**

In [7]:
history = train(model, OPTIMIZER, X_COLLOC, X_BC, U_BC, N_EPOCHS, W_DATA, W_PDE)

Epoch 00000 | Loss: 2.08e-01 | Loss data: 2.06e-01 | Loss PDE: 1.14e-03
Epoch 00100 | Loss: 1.25e-01 | Loss data: 1.19e-01 | Loss PDE: 6.08e-03
Epoch 00200 | Loss: 1.07e-01 | Loss data: 9.81e-02 | Loss PDE: 8.70e-03
Epoch 00300 | Loss: 1.02e-01 | Loss data: 9.47e-02 | Loss PDE: 7.68e-03
Epoch 00400 | Loss: 9.82e-02 | Loss data: 9.15e-02 | Loss PDE: 6.68e-03
Epoch 00500 | Loss: 9.29e-02 | Loss data: 8.71e-02 | Loss PDE: 5.76e-03
Epoch 00600 | Loss: 8.58e-02 | Loss data: 8.11e-02 | Loss PDE: 4.71e-03
Epoch 00700 | Loss: 7.83e-02 | Loss data: 7.44e-02 | Loss PDE: 3.90e-03
Epoch 00800 | Loss: 7.26e-02 | Loss data: 6.89e-02 | Loss PDE: 3.71e-03
Epoch 00900 | Loss: 6.95e-02 | Loss data: 6.58e-02 | Loss PDE: 3.62e-03
Epoch 01000 | Loss: 6.74e-02 | Loss data: 6.42e-02 | Loss PDE: 3.16e-03
Epoch 01100 | Loss: 6.58e-02 | Loss data: 6.32e-02 | Loss PDE: 2.55e-03
Epoch 01200 | Loss: 6.44e-02 | Loss data: 6.24e-02 | Loss PDE: 1.98e-03
Epoch 01300 | Loss: 6.33e-02 | Loss data: 6.17e-02 | Loss PDE: 1

### **Visualizando os resultados do treinamento**

Antes de tudo, é válido destacar como ficou nossa função de perda para esse problema:

$$
Loss = w_{data} * L_{data} + w_{PDE} * L_{PDE}
$$

que, no código, construimos dessa forma:

 ---
```python
# loss física
residual = pde_residual(model, X_colloc)
loss_pde = torch.mean(residual ** 2)

# loss dados
U_pred = model(X_bc)
loss_data = torch.mean((U_pred - U_bc) ** 2)

# loss total
loss = w_data * loss_data + w_pde * loss_pde

return loss, loss_data, loss_pde
```
---

sendo **pde_residual** a função para os cálculos do Laplaciano via *autograd* e do resíduo.

Agora, vamos dar uma olhada em como ficou a distribuição dos pontos amostrados.

In [8]:
plot_points_stationary(X_COLLOC, X_BC)

Tivemos uma amostragem com uma uniformidade mediana. Não está ruim, mas poderia ser melhor - dê uma conferida no *notebook* `07_sampling.ipynb` para saber algumas técnicas de amostragem interessantes!

Vamos ver como as perdas transcorreram no treino.

In [9]:
plot_loss(history)

Note que houve uma pequena instabilidade no início, o que seria desencorajador se tivéssemos parado antes das 10000 épocas. Após isso, ambas as perdas, a dos dados das condições de contorno e a da PDE, foram decaindo para ordens de gradeza muito baixas: $10^{-6}$ para a perda dos dados e $10^{-5}$ para a PDE! As instabilidades no meio desses ótimos valores são comuns, apenas mostram o otimizador em ação.

No geral, o modelo conseguiu aprender bem o problema, com poucos pontos amostrados no contorno e no interior.

---

Depois de olhar o treinamento, você pode estar se perguntando (eu espero): como vamos validar isso? 

### **Validando o modelo**

O bom leitor atento deve se lembrar que nosso problema possui uma solução analítica, e é ela que usaremos para avaliar nosso modelo!

Dentro da função das funções de plotagens, fizemos o seguinte:

---
```python
x = np.linspace(0, 1, n_grid)
y = np.linspace(0, 1, n_grid)
X, Y = np.meshgrid(x, y)

X_flat = torch.tensor(
    np.stack([X.ravel(), Y.ravel()], axis=1),
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    U_pred = model(X_flat).cpu().numpy().reshape(n_grid, n_grid)
    U_ref  = analytical_fn(X_flat).cpu().numpy().reshape(n_grid, n_grid)
```
---

Em suma, criamos uma malha e avaliamos nosso modelo nela. Pontos importantes para se destacar:

- Aqui, realizamos uma nova amostragem, o que é análoga a etapa de mostrar "dados nunca vistos" para o modelo. Mas note que, no treinamento usamos uma amostragem aleatória, enquanto que aqui, uma grade regular;
- Desativamos o autograd pois não utilizamos ele na previsão, dado que não atualizamos os pesos, economizando memória e acelarando a inferência.

In [10]:
results = evaluate(model, analytical_solution, DEVICE)

In [11]:
plot_heatmaps(results['U_pred'], results['U_ref'], results['x'], results['y'])

In [12]:
plot_profiles(results['U_pred_slices'], results['U_ref_slices'], results['x'], ylabel='u(x,y)')

Esses dois *plots* mostram como nosso modelo aprendeu bem! Um erro absoluto na ordem de $10^{-3}$ é excelente para nosso problema. Mas lembre, foi um exemplo simples, com condições de contorno retangulares e solução anaálitica disponível. É um bom começo!

---

> Agora, lembra da nossa discussão sobre as `Hard-PINN`, uma variação do procedimento original para PINN? É altamente recomendável prosseguir para o *notebook* `02_direct_stationary_hard.ipynb` se quiser saber mais!

---

## Referências

<a id="hands-on-paper"> </a> [BATY, Hubert. A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics. arXiv preprint arXiv:2403.00599, 2024.](https://arxiv.org/html/2403.00599v1#S2)

<a id='laplace-eq'> </a> [Wikipedia - Equação de Laplace](https://pt.wikipedia.org/wiki/Equa%C3%A7%C3%A3o_de_Laplace)